In [1]:
!pwd

/Users/zagidull/Desktop/synosys/annika-s paper/data


In [2]:
import numpy as np
import pandas as pd
import os

os.chdir('/Users/zagidull/Desktop/synosys/lina-s paper/')

In [3]:
df = pd.read_csv('data/SynoSys_27.02.2026-11.03.2026/bbi-(enhanced).csv')

In [4]:
df.head() 

,User Id,User First Name,User Last Name,User Email,Team Names,Group Names,Calendar Date (Local),Start Time (Local),Time Zone (Local),Calendar Date (UTC),Start Time (UTC),Start Time (ms),Time Zone (s),BBI (ms),Gap Duration (ms),Status,BBI Type
0,69aa9a01c272d7436ea5db59,Paultest,NaN,3aa9b47e@nomail.local,NaN,SynoSys,2026-03-06,2026-03-06T10:24:12.000,100,2026-03-06,2026-03-06T09:24:12.000,1772789052000,3600,1007,NaN,LOW_CONFIDENCE,Enhanced BBI
1,69aa9a01c272d7436ea5db59,Paultest,NaN,3aa9b47e@nomail.local,NaN,SynoSys,2026-03-06,2026-03-06T10:24:13.508,100,2026-03-06,2026-03-06T09:24:13.508,1772789053508,3600,1507,NaN,LOW_CONFIDENCE,Enhanced BBI
2,69aa9a01c272d7436ea5db59,Paultest,NaN,3aa9b47e@nomail.local,NaN,SynoSys,2026-03-06,2026-03-06T10:24:14.131,100,2026-03-06,2026-03-06T09:24:14.131,1772789054131,3600,625,NaN,LOW_CONFIDENCE,Enhanced BBI
3,69aa9a01c272d7436ea5db59,Paultest,NaN,3aa9b47e@nomail.local,NaN,SynoSys,2026-03-06,2026-03-06T10:24:14.818,100,2026-03-06,2026-03-06T09:24:14.818,1772789054818,3600,687,NaN,LOW_CONFIDENCE,Enhanced BBI
4,69aa9a01c272d7436ea5db59,Paultest,NaN,3aa9b47e@nomail.local,NaN,SynoSys,2026-03-06,2026-03-06T10:24:15.474,100,2026-03-06,2026-03-06T09:24:15.474,1772789055474,3600,656,NaN,LOW_CONFIDENCE,Enhanced BBI


In [5]:
df['User Id'].nunique()

3

In [6]:
pd.DataFrame(map(lambda col: (col, df[col].nunique(), df[col].dtype ), df.columns))

# looks like these cols are not needed rn
# 'User First Name', 'User Last Name', 'User Email', 'Team Names', 'Group Names', 
# 'Calendar Date (Local)', 'Time Zone (Local)', 'Calendar Date (UTC)', 
# 'Gap Duration (ms)', 'BBI Type'

,0,1,2
0,User Id,3,object
1,User First Name,3,object
2,User Last Name,0,float64
3,User Email,3,object
4,Team Names,0,float64
5,Group Names,1,object
6,Calendar Date (Local),6,object
7,Start Time (Local),740949,object
8,Time Zone (Local),1,int64
9,Calendar Date (UTC),6,object


In [7]:
all(
    pd.to_datetime(df['Start Time (ms)'], unit='ms', utc=True) \
    + pd.to_timedelta(df['Time Zone (s)'], unit='s') \
    == pd.to_datetime(df['Start Time (Local)'], utc=True))

# looks like Start Time (ms) is in UTC, ie Start Time (ms)+ Time Zone (s)= Start Time (Local)
# so we can drop Start Time (Local), Start Time (UTC)

True

In [8]:
cols_drop=['User First Name', 'User Last Name', 'User Email', 'Team Names', 'Group Names', 'Calendar Date (Local)',
           'Start Time (Local)', 'Start Time (UTC)', 'Time Zone (Local)', 'Calendar Date (UTC)', 'Gap Duration (ms)', 'BBI Type']

In [9]:
df = df.drop(labels=cols_drop, axis='columns', inplace=False)

In [10]:
df.head()

,User Id,Start Time (ms),Time Zone (s),BBI (ms),Status
0,69aa9a01c272d7436ea5db59,1772789052000,3600,1007,LOW_CONFIDENCE
1,69aa9a01c272d7436ea5db59,1772789053508,3600,1507,LOW_CONFIDENCE
2,69aa9a01c272d7436ea5db59,1772789054131,3600,625,LOW_CONFIDENCE
3,69aa9a01c272d7436ea5db59,1772789054818,3600,687,LOW_CONFIDENCE
4,69aa9a01c272d7436ea5db59,1772789055474,3600,656,LOW_CONFIDENCE


In [11]:
# lets map user id to something simple, like 1-2-3

userid_map = dict(zip(set(df['User Id']), {1,2,3}))
df['User Id'] = df['User Id'].map(userid_map)


In [12]:
df.Status.value_counts()

Status
LOW_CONFIDENCE     388613
HIGH_CONFIDENCE    352912
Name: count, dtype: int64

In [13]:
df.dtypes

User Id             int64
Start Time (ms)     int64
Time Zone (s)       int64
BBI (ms)            int64
Status             object
dtype: object

In [15]:
df.isna().sum()

User Id            0
Start Time (ms)    0
Time Zone (s)      0
BBI (ms)           0
Status             0
dtype: int64